# WD0 Predictions — Calibration Tuning (Stage 3)

**Goal:** Replace the predecessor's hand-tuned weight table in `Wd0PredictiveService` with data-driven, per-segment bounds that achieve ~80% empirical coverage with the tightest interval width.

## Approach

The CSV (`Result_47.csv`) contains the predecessor's stored predictions joined to ground-truth actuals — `(pred_min, pred_max, actual)` per `(FISCAL_PERIOD, WD, PRODUCT_TYPE)`. It does **not** contain the underlying `raw_sum` from the source queries.

So we fit **calibration multipliers** rather than raw weights:

1. Define `center = (pred_min + pred_max) / 2` as the signal.
2. Compute `ratio = actual / center` per row.
3. Per segment, fit `low_mult = quantile_10(ratio)`, `high_mult = quantile_90(ratio)`.
4. New bounds at prediction time: `[low_mult × center, high_mult × center]`.

**Segments tested:** 1-model (pooled), 2-model (by PRODUCT_TYPE), 4-model (PRODUCT_TYPE × close_type {ME, QE}). WD-step is held as a feature within each segment.

## Inputs / Outputs

- Input: `Result_47.csv` — 126 rows with actuals + 4 in-flight rows (dropped).
- Output: `wd0_weights_v2.json` + a Java code snippet for `Wd0PredictiveService`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

CSV_PATH = Path('Result_47.csv')
OUT_JSON = Path('wd0_weights_v2.json')

COLS = ['fiscal_period', 'wd', 'product_type', 'pred_min', 'pred_max',
        'run_date', 'actual', 'execution_time', 'period_year', 'period_num']

df = pd.read_csv(CSV_PATH, header=None, names=COLS)
print(f'raw rows: {len(df)}')
df.head()

raw rows: 130


,fiscal_period,wd,product_type,pred_min,pred_max,run_date,actual,execution_time,period_year,period_num
0,AUG-25,WD-2,PRODUCT,18152,52771,2024-10-09 09:44:17,"31,455.000",4.450,"2,025.000",1.000
1,AUG-25,WD-3,PRODUCT,12137,60183,2024-10-09 09:44:17,"31,455.000",4.450,"2,025.000",1.000
2,SEP-25,WD-2,PRODUCT,17184,52771,2024-10-09 09:44:19,"29,628.000",3.700,"2,025.000",2.000
3,SEP-25,WD-3,PRODUCT,9864,57698,2024-10-09 09:44:19,"29,628.000",3.700,"2,025.000",2.000
4,SEP-25,WD-1,PRODUCT,21332,41032,2024-10-09 09:44:32,"29,628.000",3.700,"2,025.000",2.000


## Clean & enrich

In [2]:
EXCLUDE_PERIODS = {'JUL-23', 'APR-23'}

before = len(df)
df = df.dropna(subset=['actual']).copy()
df = df[~df['fiscal_period'].isin(EXCLUDE_PERIODS)]
print(f'dropped {before - len(df)} rows (in-flight + excluded periods)')

df['close_type'] = np.where(df['period_num'].isin([3, 6, 9, 12]), 'QE', 'ME')
df['wd_step'] = df['wd'].str.replace('WD-', '').astype(int)
df['center'] = (df['pred_min'] + df['pred_max']) / 2
df['ratio'] = df['actual'] / df['center']
df['pred_width'] = df['pred_max'] - df['pred_min']
df['inside_pred'] = (df['actual'] >= df['pred_min']) & (df['actual'] <= df['pred_max'])

print(f'usable rows: {len(df)}  |  periods: {df.fiscal_period.nunique()}')
df[['fiscal_period', 'wd', 'product_type', 'close_type',
    'pred_min', 'pred_max', 'actual', 'ratio', 'inside_pred']].head()

dropped 4 rows (in-flight + excluded periods)
usable rows: 126  |  periods: 21


,fiscal_period,wd,product_type,close_type,pred_min,pred_max,actual,ratio,inside_pred
0,AUG-25,WD-2,PRODUCT,ME,18152,52771,"31,455.000",0.887,True
1,AUG-25,WD-3,PRODUCT,ME,12137,60183,"31,455.000",0.870,True
2,SEP-25,WD-2,PRODUCT,ME,17184,52771,"29,628.000",0.847,True
3,SEP-25,WD-3,PRODUCT,ME,9864,57698,"29,628.000",0.877,True
4,SEP-25,WD-1,PRODUCT,ME,21332,41032,"29,628.000",0.950,True


## Baseline — predecessor performance today

In [3]:
def baseline_report(d, group_cols):
    return (d.groupby(group_cols)
             .agg(n=('actual', 'size'),
                  coverage=('inside_pred', 'mean'),
                  mean_width=('pred_width', 'mean'),
                  median_ratio=('ratio', 'median'),
                  q10_ratio=('ratio', lambda r: np.quantile(r, 0.10)),
                  q90_ratio=('ratio', lambda r: np.quantile(r, 0.90)))
             .round(3))

print('--- Baseline (predecessor) by close_type x product_type ---')
baseline_report(df, ['close_type', 'product_type'])

--- Baseline (predecessor) by close_type x product_type ---


n  coverage  mean_width  median_ratio  q10_ratio  \
close_type product_type                                                      
ME         PRODUCT       42     0.810  13,832.976         0.897      0.707   
           SERVICE       42     0.548   3,811.905         0.971      0.250   
QE         PRODUCT       21     0.381  19,339.190         0.547      0.358   
           SERVICE       21     0.381   7,662.762         1.999      0.791   

                         q90_ratio  
close_type product_type             
ME         PRODUCT           1.822  
           SERVICE           3.525  
QE         PRODUCT           2.843  
           SERVICE           6.165

In [4]:
print('--- Baseline by close_type x product_type x wd ---')
baseline_report(df, ['close_type', 'product_type', 'wd'])

--- Baseline by close_type x product_type x wd ---


n  coverage  mean_width  median_ratio  \
close_type product_type wd                                             
ME         PRODUCT      WD-1  14     0.643   8,331.571         0.956   
                        WD-2  14     0.857  14,084.071         0.897   
                        WD-3  14     0.929  19,083.286         0.864   
           SERVICE      WD-1  14     0.571   2,388.786         1.049   
                        WD-2  14     0.500   4,138.643         1.048   
                        WD-3  14     0.571   4,908.286         0.873   
QE         PRODUCT      WD-1   7     0.000  11,606.286         0.537   
                        WD-2   7     0.429  19,543.286         0.547   
                        WD-3   7     0.714  26,868.000         0.555   
           SERVICE      WD-1   7     0.286   6,526.429         1.877   
                        WD-2   7     0.429   8,203.000         1.999   
                        WD-3   7     0.429   8,258.857         2.535   

                              q10_ratio  q90_ratio  
close_type product_type wd                          
ME         PRODUCT      WD-1      0.782      1.873  
                        WD-2      0.711      1.716  
                        WD-3      0.669      1.478  
           SERVICE      WD-1      0.608      3.364  
                        WD-2      0.292      2.983  
                        WD-3      0.300      3.656  
QE         PRODUCT      WD-1      0.440      2.474  
                        WD-2      0.425      2.454  
                        WD-3      0.379      2.262  
           SERVICE      WD-1      0.694      4.006  
                        WD-2      1.057      4.668  
                        WD-3      1.067      4.792

## Fit calibration multipliers — three architectures

In [5]:
LOW_Q, HIGH_Q = 0.10, 0.90

def fit_multipliers(d):
    r = d['ratio'].values
    return float(np.quantile(r, LOW_Q)), float(np.quantile(r, HIGH_Q))

def apply_and_score(train, test, group_cols):
    mults = (train.groupby(group_cols)
                  .apply(lambda g: pd.Series(fit_multipliers(g),
                                             index=['low_mult', 'high_mult']),
                         include_groups=False)
                  .reset_index())
    scored = test.merge(mults, on=group_cols, how='left')
    g_low, g_high = fit_multipliers(train)
    scored['low_mult'] = scored['low_mult'].fillna(g_low)
    scored['high_mult'] = scored['high_mult'].fillna(g_high)
    scored['new_low'] = scored['low_mult'] * scored['center']
    scored['new_high'] = scored['high_mult'] * scored['center']
    scored['covered'] = (scored['actual'] >= scored['new_low']) & (scored['actual'] <= scored['new_high'])
    scored['new_width'] = scored['new_high'] - scored['new_low']
    return scored

def loo_period_cv(d, group_cols):
    out = []
    for p in d['fiscal_period'].unique():
        train = d[d['fiscal_period'] != p]
        test = d[d['fiscal_period'] == p]
        out.append(apply_and_score(train, test, group_cols))
    return pd.concat(out, ignore_index=True)

ARCHITECTURES = {
    '1-model (pooled, WD only)':                ['wd'],
    '2-model (product_type x WD)':              ['product_type', 'wd'],
    '4-model (close_type x product_type x WD)': ['close_type', 'product_type', 'wd'],
}

summary = []
for name, gcols in ARCHITECTURES.items():
    scored = loo_period_cv(df, gcols)
    summary.append({
        'architecture': name,
        'coverage': scored['covered'].mean(),
        'mean_width': scored['new_width'].mean(),
        'median_width': scored['new_width'].median(),
        'segments': scored.groupby(gcols).ngroups,
    })

summary_df = pd.DataFrame(summary).round(3)
baseline_width = df['pred_width'].mean()
baseline_coverage = df['inside_pred'].mean()
print(f'BASELINE (predecessor): coverage={baseline_coverage:.3f}  mean_width={baseline_width:,.0f}\n')
summary_df

BASELINE (predecessor): coverage=0.579  mean_width=10,382


,architecture,coverage,mean_width,median_width,segments
0,"1-model (pooled, WD only)",0.746,"23,757.693","18,526.384",3
1,2-model (product_type x WD),0.722,"17,499.548","12,735.965",6
2,4-model (close_type x product_type x WD),0.690,"17,308.187","9,707.552",12


## Pick winner & inspect per-segment behavior

In [6]:
candidates = summary_df[summary_df['coverage'] >= 0.80]
if candidates.empty:
    winner_row = summary_df.sort_values('coverage', ascending=False).iloc[0]
    print(f"NOTE: no architecture hit 0.80 coverage; picking best coverage: {winner_row['architecture']}")
else:
    winner_row = candidates.sort_values('mean_width').iloc[0]
    print(f"WINNER: {winner_row['architecture']}")

WINNER_NAME = winner_row['architecture']
WINNER_COLS = ARCHITECTURES[WINNER_NAME]
print(f'  coverage = {winner_row["coverage"]:.3f}')
print(f'  mean_width = {winner_row["mean_width"]:,.0f}  (vs baseline {baseline_width:,.0f})')
print(f'  segments = {winner_row["segments"]}')

NOTE: no architecture hit 0.80 coverage; picking best coverage: 1-model (pooled, WD only)
  coverage = 0.746
  mean_width = 23,758  (vs baseline 10,382)
  segments = 3


In [7]:
final_weights = (df.groupby(WINNER_COLS)
                   .apply(lambda g: pd.Series({
                       'n': len(g),
                       'low_mult': fit_multipliers(g)[0],
                       'high_mult': fit_multipliers(g)[1],
                       'median_actual': g['actual'].median(),
                   }), include_groups=False)
                   .reset_index())
final_weights

,wd,n,low_mult,high_mult,median_actual
0,WD-1,42.000,0.538,2.869,"8,235.000"
1,WD-2,42.000,0.474,3.129,"8,235.000"
2,WD-3,42.000,0.472,3.199,"8,235.000"


In [8]:
scored = loo_period_cv(df, WINNER_COLS)
perf = (scored.groupby(WINNER_COLS)
              .agg(n=('actual', 'size'),
                   coverage=('covered', 'mean'),
                   mean_width=('new_width', 'mean'),
                   baseline_width=('pred_width', 'mean'))
              .round(3))
perf['width_reduction_pct'] = (100 * (1 - perf['mean_width'] / perf['baseline_width'])).round(1)
perf

,n,coverage,mean_width,baseline_width,width_reduction_pct
wd,,,,,
WD-1,42,0.738,"21,424.573","6,595.571",-224.800
WD-2,42,0.762,"23,745.177","10,698.619",-121.900
WD-3,42,0.738,"26,103.328","13,851.667",-88.400


## Sanity check — biggest misses

In [9]:
misses = scored[~scored['covered']].copy()
misses['miss_pct'] = np.where(
    misses['actual'] > misses['new_high'],
    (misses['actual'] - misses['new_high']) / misses['new_high'] * 100,
    (misses['new_low'] - misses['actual']) / misses['new_low'] * 100,
)
print(f'{len(misses)} misses out of {len(scored)} ({len(misses)/len(scored):.1%})')
misses[['fiscal_period', 'wd', 'product_type', 'close_type',
        'new_low', 'actual', 'new_high', 'miss_pct']].sort_values('miss_pct', key=abs, ascending=False).head(10)

32 misses out of 126 (25.4%)


,fiscal_period,wd,product_type,close_type,new_low,actual,new_high,miss_pct
29,DEC-25,WD-1,SERVICE,ME,391.972,"10,416.000","1,905.830",446.534
27,DEC-25,WD-2,SERVICE,ME,360.319,"10,416.000","2,045.604",409.189
25,DEC-25,WD-3,SERVICE,ME,376.673,"10,416.000","2,325.006",347.999
86,OCT-26,WD-2,SERVICE,QE,"1,326.321","18,565.000","7,529.791",146.554
88,OCT-26,WD-1,SERVICE,QE,"1,646.874","18,565.000","7,830.024",137.100
84,OCT-26,WD-3,SERVICE,QE,"1,282.044","18,565.000","7,913.380",134.603
82,SEP-26,WD-1,SERVICE,ME,"4,008.223","1,538.000","21,719.761",61.629
80,SEP-26,WD-2,SERVICE,ME,"3,573.941","1,538.000","22,408.840",56.966
111,FEB-26,WD-2,SERVICE,ME,"2,526.064","1,093.000","15,838.583",56.731
108,FEB-26,WD-3,SERVICE,ME,"2,401.257","1,093.000","16,130.307",54.482


## Emit JSON + Java snippet

In [10]:
payload = {
    'model_version': 'v2-calibrated',
    'architecture': WINNER_NAME,
    'group_cols': WINNER_COLS,
    'quantiles': {'low': LOW_Q, 'high': HIGH_Q},
    'training_rows': int(len(df)),
    'training_periods': sorted(df['fiscal_period'].unique().tolist()),
    'loo_cv': {
        'coverage': float(winner_row['coverage']),
        'mean_width': float(winner_row['mean_width']),
        'baseline_coverage': float(baseline_coverage),
        'baseline_mean_width': float(baseline_width),
    },
    'weights': [
        {**{c: row[c] for c in WINNER_COLS},
         'low_mult': float(row['low_mult']),
         'high_mult': float(row['high_mult']),
         'n': int(row['n'])}
        for _, row in final_weights.iterrows()
    ],
}

OUT_JSON.write_text(json.dumps(payload, indent=2))
print(f'wrote {OUT_JSON} ({len(payload["weights"])} weight rows)')

wrote wd0_weights_v2.json (3 weight rows)


In [11]:
lines = ['// Generated from notebooks/wd0_weight_tuning.ipynb',
         f'// architecture: {WINNER_NAME}',
         f'// LOO CV: coverage={winner_row["coverage"]:.3f}, mean_width={winner_row["mean_width"]:,.0f}',
         f'// Baseline:  coverage={baseline_coverage:.3f}, mean_width={baseline_width:,.0f}',
         'private static final Map<String, double[]> CALIBRATION = Map.ofEntries(']
for _, row in final_weights.iterrows():
    key = '|'.join(str(row[c]) for c in WINNER_COLS)
    lines.append(f'    Map.entry("{key}", new double[] {{ {row["low_mult"]:.4f}, {row["high_mult"]:.4f} }}),')
lines[-1] = lines[-1].rstrip(',')
lines.append(');')
print('\n'.join(lines))

// Generated from notebooks/wd0_weight_tuning.ipynb
// architecture: 1-model (pooled, WD only)
// LOO CV: coverage=0.746, mean_width=23,758
// Baseline:  coverage=0.579, mean_width=10,382
private static final Map<String, double[]> CALIBRATION = Map.ofEntries(
    Map.entry("WD-1", new double[] { 0.5384, 2.8691 }),
    Map.entry("WD-2", new double[] { 0.4737, 3.1292 }),
    Map.entry("WD-3", new double[] { 0.4720, 3.1986 })
);
